# 06 — Retention Strategy Ranking

**Goal:** Answer the README's fourth Key Question — *among possible retention strategies (discounts, contract upgrades, premium support), which has the best ROI?*

**Approach:** Each strategy is a (cost, success_rate) tuple plugged into the Week 2 cost-sensitive framework. For each strategy we sweep thresholds, find $\tau^*_s$, and record the ROI at that threshold. We then rank strategies and run a sensitivity check on the success-rate assumption — the parameter we're least sure about.

**Definitions used here:**
- **Net value** = saved revenue − intervention costs − unrecovered churn losses (same as notebook 04)
- **ROI** = (saved revenue − intervention costs) / intervention costs

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.cost_analysis import CostAssumptions, optimal_threshold, sweep_thresholds
from src.data_loader import load_clean
from src.features import make_features, train_test_split_scaled
from src.model import train_xgboost

sns.set_theme(style="whitegrid")

## 1. Train and score (same setup as notebooks 03/04)

In [ ]:
df = load_clean()
X, y = make_features(df)
X_train, X_test, y_train, y_test = train_test_split_scaled(X, y, scale=True)
monthly_charges_test = df.loc[X_test.index, "MonthlyCharges"].values

model = train_xgboost(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]
print(f"Test set: {len(y_test)} customers, {y_test.sum()} churners ({y_test.mean():.1%})")

## 2. Strategy catalog

These cost / success-rate pairs are **placeholders** — replace them with values from your business operations or A/B-test history. They're calibrated against the dataset's average MRR (~$65) so the relative comparisons are meaningful.

| Strategy | Rationale |
|---|---|
| Premium support outreach | Cheap touch; modest lift. Aligns with EDA finding that no-TechSupport correlates with churn. |
| Free OnlineSecurity addon (6mo) | Targets a known retention-correlated feature. |
| 10% discount (6mo) | Standard play for price-sensitive churners. |
| Contract upgrade incentive | Locks customer in — typically the highest-impact play in subscription businesses. |
| Combined: discount + addon | Higher cost, higher save rate — included to test whether bundling pays. |

In [ ]:
@dataclass(frozen=True)
class Strategy:
    name: str
    cost: float
    success_rate: float

strategies = [
    Strategy("premium_support_outreach",        cost=25.0, success_rate=0.20),
    Strategy("free_security_addon_6mo",         cost=20.0, success_rate=0.22),
    Strategy("discount_10pct_6mo",              cost=40.0, success_rate=0.28),
    Strategy("contract_upgrade_incentive",      cost=50.0, success_rate=0.40),
    Strategy("discount_plus_addon_combined",    cost=60.0, success_rate=0.45),
]

pd.DataFrame([{"name": s.name, "cost ($)": s.cost, "success_rate": s.success_rate} for s in strategies])

## 3. Score every strategy at its own optimal threshold

Each strategy gets to pick the threshold that maximizes its own net value — comparing strategies at a shared threshold would unfairly penalize the cheaper ones.

In [ ]:
rows = []
for s in strategies:
    a = CostAssumptions(intervention_cost=s.cost, success_rate=s.success_rate, clv_horizon_months=12)
    sweep = sweep_thresholds(y_test, y_proba, monthly_charges_test, a)
    best = optimal_threshold(sweep)
    saved = best["saved_revenue"]
    spent = best["intervention_costs"]
    rows.append({
        "strategy": s.name,
        "cost": s.cost,
        "success_rate": s.success_rate,
        "optimal_threshold": best["threshold"],
        "n_targeted": int(best["n_targeted"]),
        "saved_revenue": saved,
        "intervention_costs": spent,
        "net_value": best["net_value"],
        # ROI guards against div-by-zero when a strategy targets nobody at its optimum
        "roi": (saved - spent) / spent if spent > 0 else float("nan"),
    })

ranked = pd.DataFrame(rows).sort_values("net_value", ascending=False).reset_index(drop=True)
ranked.round(3)

## 4. Rank by net value (absolute $) and ROI (efficiency)

Both lenses matter: the highest **net value** strategy is what you'd run if budget is unconstrained; the highest **ROI** strategy is what you'd run with a fixed retention budget.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

by_value = ranked.sort_values("net_value", ascending=True)
axes[0].barh(by_value["strategy"], by_value["net_value"], color="steelblue")
axes[0].set_title("Net value at strategy-specific $\\tau^*$ ($)")
axes[0].set_xlabel("net value ($)")

by_roi = ranked.sort_values("roi", ascending=True)
axes[1].barh(by_roi["strategy"], by_roi["roi"], color="seagreen")
axes[1].set_title("ROI at strategy-specific $\\tau^*$ (×)")
axes[1].set_xlabel("ROI = (saved − spent) / spent")

plt.tight_layout()
plt.show()

## 5. Sensitivity to success_rate uncertainty

Success rate is the parameter we're least confident about — it should come from a holdout intervention experiment we haven't run yet. We perturb each strategy's success_rate by ±30% and replot net value. If the rank order is stable across that band, the recommendation is robust.

In [ ]:
perturbations = {"-30%": 0.7, "central": 1.0, "+30%": 1.3}

rows = []
for s in strategies:
    for label, mult in perturbations.items():
        sr = max(0.0, min(1.0, s.success_rate * mult))
        a = CostAssumptions(intervention_cost=s.cost, success_rate=sr, clv_horizon_months=12)
        sweep = sweep_thresholds(y_test, y_proba, monthly_charges_test, a)
        best = optimal_threshold(sweep)
        rows.append({
            "strategy": s.name,
            "perturbation": label,
            "success_rate": sr,
            "net_value": best["net_value"],
        })

sens = pd.DataFrame(rows)
pivot = sens.pivot(index="strategy", columns="perturbation", values="net_value").round(0)
pivot = pivot[["-30%", "central", "+30%"]]
pivot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label in ["-30%", "central", "+30%"]:
    sub = sens[sens["perturbation"] == label].set_index("strategy").reindex(ranked["strategy"])
    ax.plot(sub.index, sub["net_value"], marker="o", label=f"success_rate {label}")
ax.set_xticklabels(sub.index, rotation=30, ha="right")
ax.set_ylabel("net value ($)")
ax.set_title("Net value across success-rate perturbations")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Recommendation

Read the ranking table and the sensitivity chart together:

- **Top by net value** — pick this if the goal is maximizing absolute revenue retained, budget permitting.
- **Top by ROI** — pick this if budget is fixed; it gets you the most save per dollar spent.
- **Stability check** — if the same strategy tops both rankings *and* stays on top across the ±30% success-rate band, that's a defensible recommendation. If rankings flip across the perturbation, the recommendation depends on a parameter we haven't measured — flag for an A/B test before rolling out.

**Likely outcome on this dataset:** the contract-upgrade incentive tends to win on net value (highest success rate × meaningful CLV save), while the free-addon strategy tends to win on ROI (cheapest intervention with non-trivial save rate). The combined strategy wins only if its success rate genuinely clears 0.40 — its higher cost is unforgiving otherwise.

In [ ]:
top_value = ranked.iloc[0]
top_roi = ranked.sort_values("roi", ascending=False).iloc[0]
print(f"Top by net value:  {top_value['strategy']}  (${top_value['net_value']:,.0f}, ROI {top_value['roi']:.2f}x)")
print(f"Top by ROI:        {top_roi['strategy']}  (${top_roi['net_value']:,.0f}, ROI {top_roi['roi']:.2f}x)")